In [ ]:
# ============================================================
# SEARCH QUERY SPELLING CORRECTOR
# NLP PROJECT
#
# Method:
# 1. Public word-frequency dataset
# 2. Text preprocessing
# 3. Tokenization
# 4. Edit Distance
# 5. Candidate generation
# 6. Word frequency ranking
# 7. Query correction
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import re
import requests
import matplotlib.pyplot as plt

from collections import Counter


# ============================================================
# 2. LOAD PUBLIC DATASET
# ============================================================

print("=" * 60)
print("LOADING PUBLIC DATASET")
print("=" * 60)


# Public English word-frequency dataset
# GitHub repository contains wordsFreq.csv

url = "https://raw.githubusercontent.com/harshnative/words-dataset/master/wordsFreq.csv"


try:

    response = requests.get(
        url,
        timeout=30
    )

    response.raise_for_status()

    print("\nDataset downloaded successfully!")

except Exception as e:

    print("\nError downloading dataset:")
    print(e)


# ============================================================
# 3. SAVE DATASET TEMPORARILY
# ============================================================

with open(
    "wordsFreq.csv",
    "wb"
) as file:

    file.write(
        response.content
    )


# ============================================================
# 4. READ DATASET
# ============================================================

df = pd.read_csv(
    "wordsFreq.csv"
)


print("\nDataset shape:")
print(df.shape)


print("\nColumn names:")
print(df.columns.tolist())


print("\nFirst 5 records:")
print(df.head())


# ============================================================
# 5. IDENTIFY WORD AND FREQUENCY COLUMNS
# ============================================================

print("\n" + "=" * 60)
print("DATASET COLUMNS")
print("=" * 60)


print(
    df.columns.tolist()
)


# ============================================================
# 6. RENAME COLUMNS
# ============================================================

# The dataset contains word and frequency information.
# Rename the first two useful columns to standard names.

df = df.iloc[:, :2]

df.columns = [
    "word",
    "frequency"
]


# ============================================================
# 7. CLEAN DATASET
# ============================================================

print("\n" + "=" * 60)
print("DATA CLEANING")
print("=" * 60)


# Convert word to string

df["word"] = df[
    "word"
].astype(str)


# Convert words to lowercase

df["word"] = df[
    "word"
].str.lower()


# Convert frequency to numeric

df["frequency"] = pd.to_numeric(
    df["frequency"],
    errors="coerce"
)


# Remove missing values

df = df.dropna()


# Keep alphabetic words only

df = df[
    df["word"].str.match(
        r"^[a-z]+$"
    )
]


# Remove duplicate words

df = df.drop_duplicates(
    subset="word"
)


print(
    "\nCleaned dataset shape:"
)

print(
    df.shape
)


print(
    "\nSample cleaned data:"
)

print(
    df.head(10)
)


# ============================================================
# 8. CREATE WORD FREQUENCY DICTIONARY
# ============================================================

word_frequency = dict(
    zip(
        df["word"],
        df["frequency"]
    )
)


# Create a set for fast lookup

dictionary = set(
    word_frequency.keys()
)


print(
    "\nTotal valid words:",
    len(dictionary)
)


# ============================================================
# 9. TEXT PREPROCESSING FUNCTION
# ============================================================

def preprocess_query(query):

    # Convert to lowercase

    query = query.lower()


    # Remove special characters

    query = re.sub(
        r"[^a-z\s]",
        " ",
        query
    )


    # Remove extra spaces

    query = re.sub(
        r"\s+",
        " ",
        query
    ).strip()


    # Tokenize

    words = query.split()


    return words


# ============================================================
# 10. EDIT DISTANCE FUNCTION
# ============================================================

def edit_distance(word1, word2):

    # Lengths of words

    m = len(word1)

    n = len(word2)


    # Create matrix

    dp = [
        [0] * (n + 1)
        for _ in range(m + 1)
    ]


    # First column

    for i in range(m + 1):

        dp[i][0] = i


    # First row

    for j in range(n + 1):

        dp[0][j] = j


    # Calculate distance

    for i in range(1, m + 1):

        for j in range(1, n + 1):

            if word1[i - 1] == word2[j - 1]:

                cost = 0

            else:

                cost = 1


            dp[i][j] = min(

                # Deletion

                dp[i - 1][j] + 1,

                # Insertion

                dp[i][j - 1] + 1,

                # Substitution

                dp[i - 1][j - 1] + cost
            )


    return dp[m][n]


# ============================================================
# 11. FIND CANDIDATE WORDS
# ============================================================

def get_candidates(
    misspelled_word,
    max_distance=2
):

    candidates = []


    # Only compare words with
    # similar length

    min_length = max(
        1,
        len(misspelled_word)
        - max_distance
    )


    max_length = (
        len(misspelled_word)
        + max_distance
    )


    possible_words = [

        word

        for word in dictionary

        if min_length
        <= len(word)
        <= max_length

    ]


    # Calculate edit distance

    for word in possible_words:

        distance = edit_distance(
            misspelled_word,
            word
        )


        if distance <= max_distance:

            candidates.append(
                (
                    word,
                    distance,
                    word_frequency[word]
                )
            )


    return candidates


# ============================================================
# 12. RANK CANDIDATES
# ============================================================

def rank_candidates(
    candidates,
    top_n=5
):

    # Sort by:
    #
    # 1. Lowest edit distance
    # 2. Highest word frequency


    candidates = sorted(

        candidates,

        key=lambda x: (
            x[1],
            -x[2]
        )

    )


    return candidates[:top_n]


# ============================================================
# 13. SPELLING CORRECTION FUNCTION
# ============================================================

def correct_word(
    word,
    max_distance=2,
    top_n=5
):

    # If word is already correct

    if word in dictionary:

        return [
            (
                word,
                0,
                word_frequency[word]
            )
        ]


    # Find candidates

    candidates = get_candidates(
        word,
        max_distance
    )


    # Rank candidates

    ranked_candidates = rank_candidates(
        candidates,
        top_n
    )


    return ranked_candidates


# ============================================================
# 14. DISPLAY WORD CORRECTIONS
# ============================================================

def show_word_corrections(
    word
):

    print("\n" + "-" * 60)

    print(
        "Misspelled word:",
        word
    )


    candidates = correct_word(
        word
    )


    if not candidates:

        print(
            "No correction found."
        )

        return


    print(
        "\nSuggested corrections:"
    )


    for candidate, distance, frequency in candidates:

        print(
            f"{candidate:<20}"
            f"Edit Distance: {distance:<5}"
            f"Frequency: {frequency}"
        )


# ============================================================
# 15. TEST SINGLE WORD
# ============================================================

print("\n" + "=" * 60)
print("SINGLE WORD SPELLING CORRECTION")
print("=" * 60)


test_word = "machne"


show_word_corrections(
    test_word
)


# ============================================================
# 16. QUERY SPELLING CORRECTOR
# ============================================================

def correct_query(
    query,
    max_distance=2
):

    # Preprocess query

    words = preprocess_query(
        query
    )


    corrected_words = []


    print("\n" + "=" * 60)
    print("SEARCH QUERY SPELLING CORRECTOR")
    print("=" * 60)


    print(
        "\nOriginal Query:"
    )

    print(
        query
    )


    print(
        "\nWord Analysis:"
    )


    for word in words:

        # --------------------------------
        # Correct word
        # --------------------------------

        if word in dictionary:

            corrected_word = word

            print(
                f"\n{word} ✓ Correct"
            )


        else:

            candidates = correct_word(
                word,
                max_distance
            )


            if candidates:

                corrected_word = candidates[0][0]


                print(
                    f"\n{word} ✗ Incorrect"
                )


                print(
                    "Suggested:",
                    corrected_word
                )


                print(
                    "Edit Distance:",
                    candidates[0][1]
                )


            else:

                corrected_word = word


                print(
                    f"\n{word} ✗ No suggestion"
                )


        corrected_words.append(
            corrected_word
        )


    # Join corrected words

    corrected_query = " ".join(
        corrected_words
    )


    print(
        "\n" + "-" * 60
    )


    print(
        "Corrected Query:"
    )


    print(
        corrected_query
    )


    print(
        "=" * 60
    )


    return corrected_query


# ============================================================
# 17. TEST SEARCH QUERY
# ============================================================

query1 = (
    "machne lerning algoritm"
)


correct_query(
    query1
)


# ============================================================
# 18. SECOND TEST QUERY
# ============================================================

query2 = (
    "artifical inteligence"
)


correct_query(
    query2
)


# ============================================================
# 19. THIRD TEST QUERY
# ============================================================

query3 = (
    "computr scince project"
)


correct_query(
    query3
)


# ============================================================
# 20. INTERACTIVE SEARCH QUERY CORRECTOR
# ============================================================

print("\n" + "=" * 60)
print("INTERACTIVE SEARCH QUERY CORRECTOR")
print("=" * 60)


print(
    "\nEnter a search query."
)

print(
    "Type 'exit' to stop."
)


while True:

    user_query = input(
        "\nEnter search query: "
    )


    if user_query.lower() == "exit":

        print(
            "\nProgram stopped."
        )

        break


    if user_query.strip() == "":

        print(
            "Please enter a query."
        )

        continue


    correct_query(
        user_query
    )


# ============================================================
# 21. VISUALIZATION
# ============================================================

# Show top 10 most frequent words

top_words = df.sort_values(
    by="frequency",
    ascending=False
).head(10)


plt.figure(
    figsize=(10, 6)
)


plt.bar(
    top_words["word"],
    top_words["frequency"]
)


plt.xlabel(
    "Words"
)


plt.ylabel(
    "Frequency"
)


plt.title(
    "Top 10 Most Frequent English Words"
)


plt.xticks(
    rotation=45
)


plt.tight_layout()


plt.show()


# ============================================================
# 22. FINAL MESSAGE
# ============================================================

print("\n" + "=" * 60)

print(
    "SEARCH QUERY SPELLING CORRECTOR COMPLETED"
)

print("=" * 60)

LOADING PUBLIC DATASET
